# <span style="color:blue">DATA PREPROCESSING</span> #

## <span style="color:blue">PACKAGES USED</span> ##

In [5]:
import pandas as pd
import numpy as np
from IPython.display import display
import kagglehub
from pathlib import Path
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import gc
import shutil


## <span style="color:blue">RAW DATA ACQUISITION AND TRANSFORMATION INTO PRIMARY DATASET</span> ##

### <span style="color:blue"> checking whether primary data already exist and downloading them if they do not </span> ###

In [6]:
# ============================================================
# 01. SETTINGS
# ============================================================


kaggle_dataset = "kartik2112/fraud-detection"

data_folder = Path(
    "/projeto_tcc_2026/data/initial_dataset"
)

train_file = data_folder / "fraudTrain.csv"
test_file = data_folder / "fraudTest.csv"


# ============================================================
# 02. ENSURE THE DATA FOLDER EXISTS
# ============================================================

data_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 03. CHECK THE ORIGINAL DATASET FILES
# ============================================================

required_files = {
    "fraudTrain.csv": train_file,
    "fraudTest.csv": test_file
}


for file_name, file_path in required_files.items():

    if file_path.exists():

        print(
            f"{file_name} already exists."
        )

        print(
            "Download not required."
        )

    else:

        print(
            f"\n{file_name} not found."
        )

        print(
            f"Downloading {file_name}..."
        )

        kagglehub.dataset_download(
            kaggle_dataset,
            path=file_name,
            output_dir=str(data_folder)
        )

        print(
            f"{file_name} downloaded successfully!"
        )


# ============================================================
# 04. CONFIRM THAT BOTH FILES EXIST
# ============================================================

for file_name, file_path in required_files.items():

    if not file_path.exists():

        raise FileNotFoundError(
            f"{file_name} was not found in "
            f"{data_folder}."
        )


print(
    "\nBoth original dataset files are available."
)


# ============================================================
# 05. LOAD THE ORIGINAL DATASETS
# ============================================================

print(
    "\nLoading the original datasets..."
)

train = pd.read_csv(
    train_file
)

test = pd.read_csv(
    test_file
)


# ============================================================
# 06. CHECK THE ORIGINAL DATASET DIMENSIONS
# ============================================================

print(
    "\nORIGINAL DATASET DIMENSIONS"
)

print(
    "=" * 100
)

print(
    "fraudTrain:",
    train.shape
)

print(
    "fraudTest: ",
    test.shape
)
# ============================================================
# 07. REMOVE THE .complete FOLDER
# ============================================================

complete_folder = data_folder / ".complete"

if complete_folder.exists():

    shutil.rmtree(
        complete_folder
    )

    print(
        "\n.complete folder removed successfully."
    )

else:

    print(
        "\n.complete folder was not found."
    )


fraudTrain.csv not found.


100%|██████████| 141M/141M [00:14<00:00, 10.1MB/s] 

Extracting zip of fraudTrain.csv...


fraudTrain.csv downloaded successfully!

fraudTest.csv not found.


100%|██████████| 60.5M/60.5M [00:05<00:00, 12.5MB/s]

Extracting zip of fraudTest.csv...


fraudTest.csv downloaded successfully!

Both original dataset files are available.

Loading the original datasets...

ORIGINAL DATASET DIMENSIONS
fraudTrain: (1296675, 23)
fraudTest:  (555719, 23)

.complete folder removed successfully.


### <span style="color:blue"> develop the primary dataset </span> ###

In [7]:
# ============================================================
# 08. DEFINE THE PRIMARY DATASET FILE
# ============================================================

primary_dataset_file = (
    data_folder / "dataset_primary.csv"
)


# ============================================================
# 09. CHECK IF THE PRIMARY DATASET ALREADY EXISTS
# ============================================================

if primary_dataset_file.exists():

    print(
        "\ndataset_primary.csv already exists."
    )

    print(
        "No action required."
    )

else:

    # ========================================================
    # 10. CHECK IF THE FEATURES ARE IDENTICAL
    # ========================================================

    if list(train.columns) == list(test.columns):

        print(
            "\nBoth datasets have "
            "exactly the same features."
        )

    else:

        print(
            "\nWARNING:"
        )

        print(
            "The datasets have differences "
            "in their features."
        )

        print(
            "\nFeatures only in Train:"
        )

        print(
            set(train.columns)
            - set(test.columns)
        )

        print(
            "\nFeatures only in Test:"
        )

        print(
            set(test.columns)
            - set(train.columns)
        )

        raise ValueError(
            "Train and Test have different structures."
        )


    # ========================================================
    # 11. REMOVE THE OLD IDENTIFIER
    # ========================================================

    if "Unnamed: 0" in train.columns:

        train = train.drop(
            columns=["Unnamed: 0"]
        )

    if "Unnamed: 0" in test.columns:

        test = test.drop(
            columns=["Unnamed: 0"]
        )


    # ========================================================
    # 12. MERGE TRAIN AND TEST
    # ========================================================

    primary_dataset = pd.concat(
        [
            train,
            test
        ],
        axis=0,
        ignore_index=True
    )


    # ========================================================
    # 13. CREATE A NEW UNIQUE IDENTIFIER
    # ========================================================

    primary_dataset.insert(
        0,
        "NID",
        range(
            1,
            len(primary_dataset) + 1
        )
    )


    # ========================================================
    # 14. CHECK THE NEW UNIQUE IDENTIFIER
    # ========================================================

    print(
        "\nNID CHECK"
    )

    print(
        "=" * 100
    )

    print(
        "First NID:",
        primary_dataset["NID"].min()
    )

    print(
        "Last NID:",
        primary_dataset["NID"].max()
    )

    print(
        "Number of unique NIDs:",
        primary_dataset["NID"].nunique()
    )

    print(
        "Number of duplicated NIDs:",
        primary_dataset["NID"]
        .duplicated()
        .sum()
    )


    # ========================================================
    # 15. CHECK THE PRIMARY DATASET DIMENSIONS
    # ========================================================

    print(
        "\nPRIMARY DATASET DIMENSIONS"
    )

    print(
        "=" * 100
    )

    print(
        "Rows:",
        primary_dataset.shape[0]
    )

    print(
        "Features:",
        primary_dataset.shape[1]
    )


    # ========================================================
    # 16. SAVE THE PRIMARY DATASET
    # ========================================================

    primary_dataset.to_csv(
        primary_dataset_file,
        index=False
    )


    # ========================================================
    # 17. CONFIRM THE FILE WAS SAVED
    # ========================================================

    if primary_dataset_file.exists():

        print(
            "\ndataset_primary.csv "
            "created successfully!"
        )

        print(
            "Location:",
            primary_dataset_file
        )

    else:

        raise FileNotFoundError(
            "dataset_primary.csv was not created."
        )


Both datasets have exactly the same features.

NID CHECK
First NID: 1
Last NID: 1852394
Number of unique NIDs: 1852394
Number of duplicated NIDs: 0

PRIMARY DATASET DIMENSIONS
Rows: 1852394
Features: 23

dataset_primary.csv created successfully!
Location: /projeto_tcc_2026/data/initial_dataset/dataset_primary.csv


### <span style="color:blue"> develop the primary dataset </span> ###

In [8]:
# ============================================================
# 01. SETTINGS
# ============================================================


primary_dataset_file = Path(
    "/projeto_tcc_2026/data/initial_dataset/dataset_primary.csv"
)

medium_data_folder = Path(
    "/projeto_tcc_2026/data/medium_dataset"
)

medium_dataset_file = (
    medium_data_folder / "dataset_medium.csv"
)


# ============================================================
# 02. ENSURE THE MEDIUM DATASET FOLDER EXISTS
# ============================================================

medium_data_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 03. CHECK IF THE MEDIUM DATASET ALREADY EXISTS
# ============================================================

if medium_dataset_file.exists():

    print(
        "\ndataset_medium.csv already exists."
    )

    print(
        "No transformation or saving is required."
    )

else:

    # ========================================================
    # 04. CHECK IF THE PRIMARY DATASET EXISTS
    # ========================================================

    if not primary_dataset_file.exists():

        raise FileNotFoundError(
            "dataset_primary.csv was not found in "
            "/projeto_tcc_2026/data/initial_dataset."
        )


    # ========================================================
    # 05. LOAD THE PRIMARY DATASET
    # ========================================================

    df = pd.read_csv(
        primary_dataset_file
    )

    print(
        "\nInitial dimensions:",
        df.shape
    )


    # ========================================================
    # 06. CHECK THE NID IDENTIFIER
    # ========================================================

    print(
        "\nINITIAL NID CHECK"
    )

    print(
        "=" * 100
    )

    print(
        "First NID:",
        df["NID"].min()
    )

    print(
        "Last NID:",
        df["NID"].max()
    )

    print(
        "Unique NIDs:",
        df["NID"].nunique()
    )

    print(
        "Duplicated NIDs:",
        df["NID"]
        .duplicated()
        .sum()
    )


    # ========================================================
    # 07. REMOVE UNUSED FEATURES
    # ========================================================

    columns_to_remove = [
        "city",
        "state",
        "zip",
        "trans_num",
        "unix_time",
        "street"
    ]

    df = df.drop(
        columns=[
            column
            for column in columns_to_remove
            if column in df.columns
        ]
    )


    # ========================================================
    # 08. RENAME THE TARGET FEATURE
    # ========================================================

    if "is_fraud" in df.columns:

        df = df.rename(
            columns={
                "is_fraud": "TARGET"
            }
        )

    elif "is_fraude" in df.columns:

        df = df.rename(
            columns={
                "is_fraude": "TARGET"
            }
        )


    # ========================================================
    # 09. PROCESS TRANSACTION DATE AND TIME
    # ========================================================

    df["trans_date_trans_time"] = pd.to_datetime(
        df["trans_date_trans_time"],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )


    # --------------------------------------------------------
    # DAY OF THE MONTH
    # --------------------------------------------------------

    df["TRANS_DAY"] = (
        df["trans_date_trans_time"]
        .dt.day
    )


    # --------------------------------------------------------
    # DAY OF THE WEEK
    # --------------------------------------------------------

    weekday_mapping = {
        0: "Monday",
        1: "Tuesday",
        2: "Wednesday",
        3: "Thursday",
        4: "Friday",
        5: "Saturday",
        6: "Sunday"
    }

    df["TRANS_WEEK"] = (
        df["trans_date_trans_time"]
        .dt.dayofweek
        .map(weekday_mapping)
    )


    # --------------------------------------------------------
    # YEAR
    # --------------------------------------------------------

    df["TRANS_YEAR"] = (
        df["trans_date_trans_time"]
        .dt.year
    )


    # --------------------------------------------------------
    # MONTH AS SINE AND COSINE
    # --------------------------------------------------------

    month = (
        df["trans_date_trans_time"]
        .dt.month
    )

    df["TRANS_MONTH_SIN"] = np.sin(
        2 * np.pi * (month - 1) / 12
    )

    df["TRANS_MONTH_COS"] = np.cos(
        2 * np.pi * (month - 1) / 12
    )


    # --------------------------------------------------------
    # TIME AS SINE AND COSINE
    # Considering hour, minute, and second
    # --------------------------------------------------------

    decimal_hour = (
        df["trans_date_trans_time"].dt.hour
        + df["trans_date_trans_time"].dt.minute / 60
        + df["trans_date_trans_time"].dt.second / 3600
    )

    df["TRANS_HOUR_SIN"] = np.sin(
        2 * np.pi * decimal_hour / 24
    )

    df["TRANS_HOUR_COS"] = np.cos(
        2 * np.pi * decimal_hour / 24
    )


    # --------------------------------------------------------
    # REMOVE THE ORIGINAL DATE AND TIME FEATURE
    # --------------------------------------------------------

    df = df.drop(
        columns=[
            "trans_date_trans_time"
        ]
    )


    # ========================================================
    # 10. RENAME TRANSACTION AND SENDER FEATURES
    # ========================================================

    df = df.rename(
        columns={
            "amt": "TRANS_VALUE",
            "cc_num": "TRANS_NUM_CARD",
            "job": "SEND_JOB",
            "gender": "SEND_GENDER",
            "lat": "SEND_LAT_REGISTER",
            "long": "SEND_LONG_REGISTER",
            "city_pop": "SEND_POP_REGISTER"
        }
    )


    # ========================================================
    # 11. CREATE THE SENDER'S FULL NAME
    # ========================================================

    df["SEND_NAME"] = (
        df["first"]
        .fillna("")
        .astype(str)
        .str.strip()
        + " "
        + df["last"]
        .fillna("")
        .astype(str)
        .str.strip()
    ).str.strip()


    # ========================================================
    # 12. CONVERT DATE OF BIRTH INTO AGE
    # ========================================================

    df["dob"] = pd.to_datetime(
        df["dob"],
        errors="coerce"
    )


    # --------------------------------------------------------
    # FIXED REFERENCE DATE
    # --------------------------------------------------------

    reference_date = pd.Timestamp(
        "2026-09-03"
    )


    # --------------------------------------------------------
    # AGE IN DECIMAL YEARS
    # --------------------------------------------------------

    df["SEND_AGE"] = (
        (
            reference_date
            - df["dob"]
        ).dt.total_seconds()
        / (
            365.2425
            * 24
            * 60
            * 60
        )
    ).round(2)


    # --------------------------------------------------------
    # REMOVE THE ORIGINAL FEATURES
    # --------------------------------------------------------

    df = df.drop(
        columns=[
            "first",
            "last",
            "dob"
        ]
    )


    # ========================================================
    # 13. RENAME RECEIVER FEATURES
    # ========================================================

    df = df.rename(
        columns={
            "merchant": "RECEIVE_LOC",
            "category": "RECEIVE_CATEGORY",
            "merch_lat": "RECEIVE_LAT",
            "merch_long": "RECEIVE_LONG"
        }
    )


    # ========================================================
    # 14. ADD FUTURE ENCODER NOMENCLATURE
    # ========================================================

    df = df.rename(
        columns={

            # ------------------------------------------------
            # IDENTIFIER
            # ------------------------------------------------

            "NID": "NID_ALPHA",

            # ------------------------------------------------
            # FUTURE FREQUENCY ENCODING WITH FALLBACK
            # ------------------------------------------------

            "TRANS_NUM_CARD": "TRANS_NUM_CARD_FEWF",
            "RECEIVE_LOC": "RECEIVE_LOC_FEWF",
            "SEND_JOB": "SEND_JOB_FEWF",
            "SEND_NAME": "SEND_NAME_FEWF",

            # ------------------------------------------------
            # FUTURE BINARY ENCODING
            # ------------------------------------------------

            "SEND_GENDER": "SEND_GENDER_BE",
            "TRANS_YEAR": "TRANS_YEAR_BE",

            # ------------------------------------------------
            # FUTURE ONE-HOT ENCODING WITH IGNORE
            # ------------------------------------------------

            "RECEIVE_CATEGORY": "RECEIVE_CATEGORY_OHEWI",
            "TRANS_WEEK": "TRANS_WEEK_OHEWI",

            # ------------------------------------------------
            # TARGET VARIABLE
            # ------------------------------------------------

            "TARGET": "TARGET_OMEGA"
        }
    )


    # ========================================================
    # 15. PLACE NID FIRST AND TARGET LAST
    # ========================================================

    middle_columns = [
        column
        for column in df.columns
        if column not in [
            "NID_ALPHA",
            "TARGET_OMEGA"
        ]
    ]

    df = df[
        ["NID_ALPHA"]
        + middle_columns
        + ["TARGET_OMEGA"]
    ]


    # ========================================================
    # 16. CHECK THE MEDIUM DATASET
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "MEDIUM DATASET"
    )

    print(
        "=" * 100
    )

    print(
        f"Rows: {df.shape[0]}"
    )

    print(
        f"Features: {df.shape[1]}"
    )


    # ========================================================
    # 17. DISPLAY THE FINAL FEATURES
    # ========================================================

    print(
        "\nFINAL FEATURES"
    )

    print(
        "=" * 100
    )

    for index, column in enumerate(
        df.columns,
        start=1
    ):

        print(
            f"{index}. {column}"
        )


    # ========================================================
    # 18. CHECK SEND_AGE
    # ========================================================

    print(
        "\nSEND_AGE CHECK"
    )

    print(
        "=" * 100
    )

    print(
        "Reference date:",
        reference_date.strftime(
            "%Y-%m-%d"
        )
    )

    print(
        "Minimum age:",
        df["SEND_AGE"].min()
    )

    print(
        "Maximum age:",
        df["SEND_AGE"].max()
    )


    # ========================================================
    # 19. CHECK THE TARGET VARIABLE
    # ========================================================

    print(
        "\nTARGET CHECK"
    )

    print(
        "=" * 100
    )

    print(
        df["TARGET_OMEGA"]
        .value_counts()
        .sort_index()
    )


    # ========================================================
    # 20. SAVE THE MEDIUM DATASET
    # ========================================================

    df.to_csv(
        medium_dataset_file,
        index=False
    )


    # ========================================================
    # 21. CONFIRM THE FILE WAS SAVED
    # ========================================================

    if medium_dataset_file.exists():

        print(
            "\ndataset_medium.csv "
            "created successfully!"
        )

        print(
            "Location:",
            medium_dataset_file
        )

    else:

        raise FileNotFoundError(
            "dataset_medium.csv was not created."
        )


Initial dimensions: (1852394, 23)

INITIAL NID CHECK
First NID: 1
Last NID: 1852394
Unique NIDs: 1852394
Duplicated NIDs: 0

MEDIUM DATASET
Rows: 1852394
Features: 22

FINAL FEATURES
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. RECEIVE_LOC_FEWF
4. RECEIVE_CATEGORY_OHEWI
5. TRANS_VALUE
6. SEND_GENDER_BE
7. SEND_LAT_REGISTER
8. SEND_LONG_REGISTER
9. SEND_POP_REGISTER
10. SEND_JOB_FEWF
11. RECEIVE_LAT
12. RECEIVE_LONG
13. TRANS_DAY
14. TRANS_WEEK_OHEWI
15. TRANS_YEAR_BE
16. TRANS_MONTH_SIN
17. TRANS_MONTH_COS
18. TRANS_HOUR_SIN
19. TRANS_HOUR_COS
20. SEND_NAME_FEWF
21. SEND_AGE
22. TARGET_OMEGA

SEND_AGE CHECK
Reference date: 2026-09-03
Minimum age: 21.59
Maximum age: 101.84

TARGET CHECK
TARGET_OMEGA
0    1842743
1       9651
Name: count, dtype: int64

dataset_medium.csv created successfully!
Location: /projeto_tcc_2026/data/medium_dataset/dataset_medium.csv


### <span style="color:blue">DATASET OPTIMIZATION FROM CSV TO PARQUET</span> ###

In [9]:
# ============================================================
# 01. SETTINGS
# ============================================================

input_file = Path(
    "/projeto_tcc_2026/data/medium_dataset/dataset_medium.csv"
)

output_folder = Path(
    "/projeto_tcc_2026/data/final_dataset"
)

output_file = (
    output_folder / "dataset_final.parquet"
)


# ============================================================
# 02. CHECK IF THE FINAL DATASET ALREADY EXISTS
# ============================================================

if output_file.exists():

    print(
        "\ndataset_final.parquet already exists."
    )

    print(
        "No optimization or saving is required."
    )

else:

    # ========================================================
    # 03. CHECK IF THE INPUT DATASET EXISTS
    # ========================================================

    if not input_file.exists():

        raise FileNotFoundError(
            f"Input file not found:\n{input_file}"
        )


    # ========================================================
    # 04. ENSURE THE OUTPUT FOLDER EXISTS
    # ========================================================

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )


    # ========================================================
    # 05. LOAD THE MEDIUM DATASET
    # ========================================================

    print(
        "\nLoading the medium dataset..."
    )

    df = pd.read_csv(
        input_file
    )

    print(
        "Dataset loaded successfully!"
    )


    # ========================================================
    # 06. RECORD INFORMATION BEFORE OPTIMIZATION
    # ========================================================

    types_before = (
        df.dtypes
        .astype(str)
        .copy()
    )

    column_memory_before = (
        df.memory_usage(
            index=False,
            deep=True
        )
        .copy()
    )

    total_memory_before = (
        column_memory_before.sum()
    )

    disk_size_before = (
        input_file.stat().st_size
    )


    # ========================================================
    # 07. OPTIMIZATION 1
    # FLOATING-POINT NUMBERS
    # float64 → float32 WHEN POSSIBLE
    # ========================================================

    print(
        "\nOptimizing floating-point numbers..."
    )

    float_columns = (
        df.select_dtypes(
            include=["floating"]
        )
        .columns
    )

    for column in float_columns:

        df[column] = pd.to_numeric(
            df[column],
            downcast="float"
        )


    # ========================================================
    # 08. OPTIMIZATION 2
    # INTEGER NUMBERS
    # int64 → int32 / int16 / int8 WHEN POSSIBLE
    # ========================================================

    print(
        "Optimizing integer numbers..."
    )

    integer_columns = (
        df.select_dtypes(
            include=["integer"]
        )
        .columns
    )

    for column in integer_columns:

        df[column] = pd.to_numeric(
            df[column],
            downcast="integer"
        )


    # ========================================================
    # 09. OPTIMIZATION 3
    # STRING / OBJECT → CATEGORY
    # ========================================================

    print(
        "Analyzing categorical features..."
    )

    text_columns = (
        df.select_dtypes(
            include=[
                "object",
                "string"
            ]
        )
        .columns
    )

    columns_converted_to_category = []


    for column in text_columns:

        unique_count = (
            df[column]
            .nunique(
                dropna=False
            )
        )

        unique_ratio = (
            unique_count
            / len(df)
        )


        # ----------------------------------------------------
        # RULE:
        # Convert to category when no more than 5% of the
        # values in the feature are unique.
        # ----------------------------------------------------

        if unique_ratio <= 0.05:

            df[column] = (
                df[column]
                .astype("category")
            )

            columns_converted_to_category.append(
                column
            )


    # ========================================================
    # 10. RECORD INFORMATION AFTER OPTIMIZATION
    # ========================================================

    types_after = (
        df.dtypes
        .astype(str)
        .copy()
    )

    column_memory_after = (
        df.memory_usage(
            index=False,
            deep=True
        )
        .copy()
    )

    total_memory_after = (
        column_memory_after.sum()
    )


    # ========================================================
    # 11. CREATE A SUMMARY BY FEATURE
    # ========================================================

    feature_summary = pd.DataFrame({

        "TYPE_BEFORE":
            types_before,

        "TYPE_AFTER":
            types_after,

        "MEMORY_BEFORE_MB":
            column_memory_before
            / 1024**2,

        "MEMORY_AFTER_MB":
            column_memory_after
            / 1024**2
    })


    # ========================================================
    # 12. CALCULATE MEMORY SAVINGS BY FEATURE
    # ========================================================

    feature_summary[
        "MEMORY_SAVED_MB"
    ] = (

        feature_summary[
            "MEMORY_BEFORE_MB"
        ]

        -

        feature_summary[
            "MEMORY_AFTER_MB"
        ]
    )


    feature_summary[
        "REDUCTION_%"
    ] = (

        feature_summary[
            "MEMORY_SAVED_MB"
        ]

        /

        feature_summary[
            "MEMORY_BEFORE_MB"
        ]

        * 100
    )


    feature_summary[
        "TYPE_CHANGED"
    ] = (

        feature_summary[
            "TYPE_BEFORE"
        ]

        !=

        feature_summary[
            "TYPE_AFTER"
        ]
    )


    # ========================================================
    # 13. SAVE THE FINAL DATASET AS PARQUET
    # ========================================================

    print(
        "\nSaving the final dataset..."
    )

    df.to_parquet(
        output_file,
        engine="pyarrow",
        compression="zstd",
        index=False
    )

    print(
        "Final dataset saved successfully!"
    )


    # ========================================================
    # 14. RECORD THE FINAL FILE SIZE
    # ========================================================

    disk_size_after = (
        output_file.stat().st_size
    )


    # ========================================================
    # 15. CALCULATE RAM REDUCTION
    # ========================================================

    memory_reduction = (

        1
        - (
            total_memory_after
            /
            total_memory_before
        )

    ) * 100


    # ========================================================
    # 16. CALCULATE DISK SPACE REDUCTION
    # ========================================================

    disk_reduction = (

        1
        - (
            disk_size_after
            /
            disk_size_before
        )

    ) * 100


    # ========================================================
    # 17. FINAL DATASET SUMMARY
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "FINAL OPTIMIZED DATASET SUMMARY"
    )

    print(
        "=" * 100
    )


    print(
        f"\nRows: "
        f"{df.shape[0]}"
    )

    print(
        f"Features: "
        f"{df.shape[1]}"
    )

    print(
        f"Missing values: "
        f"{df.isna().sum().sum()}"
    )


    # ========================================================
    # 18. RAM MEMORY
    # ========================================================

    print(
        "\n"
        + "-" * 100
    )

    print(
        "RAM MEMORY"
    )

    print(
        "-" * 100
    )


    print(
        f"Before: "
        f"{total_memory_before / 1024**2:.2f} MB"
    )

    print(
        f"After: "
        f"{total_memory_after / 1024**2:.2f} MB"
    )

    print(
        f"Memory saved: "
        f"{(total_memory_before - total_memory_after) / 1024**2:.2f} MB"
    )

    print(
        f"Reduction: "
        f"{memory_reduction:.2f}%"
    )


    # ========================================================
    # 19. DISK SPACE
    # ========================================================

    print(
        "\n"
        + "-" * 100
    )

    print(
        "DISK SPACE"
    )

    print(
        "-" * 100
    )


    print(
        f"Previous file: "
        f"{disk_size_before / 1024**2:.2f} MB"
    )

    print(
        f"Final file: "
        f"{disk_size_after / 1024**2:.2f} MB"
    )

    print(
        f"Disk space saved: "
        f"{(disk_size_before - disk_size_after) / 1024**2:.2f} MB"
    )

    print(
        f"Reduction: "
        f"{disk_reduction:.2f}%"
    )


    # ========================================================
    # 20. DATA TYPES BEFORE OPTIMIZATION
    # ========================================================

    print(
        "\n"
        + "-" * 100
    )

    print(
        "DATA TYPES BEFORE OPTIMIZATION"
    )

    print(
        "-" * 100
    )

    display(
        types_before
        .value_counts()
        .rename("COUNT")
        .to_frame()
    )


    # ========================================================
    # 21. DATA TYPES AFTER OPTIMIZATION
    # ========================================================

    print(
        "\n"
        + "-" * 100
    )

    print(
        "DATA TYPES AFTER OPTIMIZATION"
    )

    print(
        "-" * 100
    )

    display(
        types_after
        .value_counts()
        .rename("COUNT")
        .to_frame()
    )


    # ========================================================
    # 22. FEATURES WITH CHANGED DATA TYPES
    # ========================================================

    print(
        "\n"
        + "-" * 100
    )

    print(
        "FEATURES WITH CHANGED DATA TYPES"
    )

    print(
        "-" * 100
    )

    display(

        feature_summary[
            feature_summary[
                "TYPE_CHANGED"
            ]
        ]

        .sort_values(
            "MEMORY_SAVED_MB",
            ascending=False
        )

        .round(2)
    )


    # ========================================================
    # 23. COMPLETE FEATURE SUMMARY
    # ========================================================

    print(
        "\n"
        + "-" * 100
    )

    print(
        "COMPLETE FEATURE SUMMARY"
    )

    print(
        "-" * 100
    )

    display(

        feature_summary
        .sort_values(
            "MEMORY_BEFORE_MB",
            ascending=False
        )
        .round(2)
    )


    # ========================================================
    # 24. FEATURES CONVERTED TO CATEGORY
    # ========================================================

    print(
        "\n"
        + "-" * 100
    )

    print(
        "FEATURES CONVERTED TO CATEGORY"
    )

    print(
        "-" * 100
    )


    if columns_converted_to_category:

        for column in columns_converted_to_category:

            print(
                f"- {column}"
            )

    else:

        print(
            "No features were converted to category."
        )


    # ========================================================
    # 25. CONFIRM THE FINAL DATASET
    # ========================================================

    if output_file.exists():

        print(
            "\n"
            + "=" * 100
        )

        print(
            "FINAL DATASET"
        )

        print(
            "=" * 100
        )

        print(
            "dataset_final.parquet "
            "created successfully!"
        )

        print(
            "Location:",
            output_file
        )

    else:

        raise FileNotFoundError(
            "dataset_final.parquet was not created."
        )


Loading the medium dataset...
Dataset loaded successfully!

Optimizing floating-point numbers...
Optimizing integer numbers...
Analyzing categorical features...

Saving the final dataset...
Final dataset saved successfully!

FINAL OPTIMIZED DATASET SUMMARY

Rows: 1852394
Features: 22
Missing values: 0

----------------------------------------------------------------------------------------------------
RAM MEMORY
----------------------------------------------------------------------------------------------------
Before: 443.47 MB
After: 129.02 MB
Memory saved: 314.45 MB
Reduction: 70.91%

----------------------------------------------------------------------------------------------------
DISK SPACE
----------------------------------------------------------------------------------------------------
Previous file: 435.41 MB
Final file: 49.73 MB
Disk space saved: 385.68 MB
Reduction: 88.58%

--------------------------------------------------------------------------------------------------

,COUNT
float64,10
int64,6
str,6



----------------------------------------------------------------------------------------------------
DATA TYPES AFTER OPTIMIZATION
----------------------------------------------------------------------------------------------------


,COUNT
float32,9
category,6
int32,2
int8,2
int64,1
float64,1
int16,1



----------------------------------------------------------------------------------------------------
FEATURES WITH CHANGED DATA TYPES
----------------------------------------------------------------------------------------------------


,TYPE_BEFORE,TYPE_AFTER,MEMORY_BEFORE_MB,MEMORY_AFTER_MB,MEMORY_SAVED_MB,REDUCTION_%,TYPE_CHANGED
RECEIVE_LOC_FEWF,str,category,54.99,3.55,51.44,93.54,True
SEND_JOB_FEWF,str,category,49.87,3.55,46.33,92.89,True
SEND_NAME_FEWF,str,category,37.44,3.55,33.89,90.51,True
RECEIVE_CATEGORY_OHEWI,str,category,32.73,1.77,30.96,94.60,True
TRANS_WEEK_OHEWI,str,category,26.41,1.77,24.65,93.31,True
SEND_GENDER_BE,str,category,15.90,1.77,14.13,88.89,True
TARGET_OMEGA,int64,int8,14.13,1.77,12.37,87.50,True
TRANS_DAY,int64,int8,14.13,1.77,12.37,87.50,True
TRANS_YEAR_BE,int64,int16,14.13,3.53,10.60,75.00,True
NID_ALPHA,int64,int32,14.13,7.07,7.07,50.00,True



----------------------------------------------------------------------------------------------------
COMPLETE FEATURE SUMMARY
----------------------------------------------------------------------------------------------------


,TYPE_BEFORE,TYPE_AFTER,MEMORY_BEFORE_MB,MEMORY_AFTER_MB,MEMORY_SAVED_MB,REDUCTION_%,TYPE_CHANGED
RECEIVE_LOC_FEWF,str,category,54.99,3.55,51.44,93.54,True
SEND_JOB_FEWF,str,category,49.87,3.55,46.33,92.89,True
SEND_NAME_FEWF,str,category,37.44,3.55,33.89,90.51,True
RECEIVE_CATEGORY_OHEWI,str,category,32.73,1.77,30.96,94.60,True
TRANS_WEEK_OHEWI,str,category,26.41,1.77,24.65,93.31,True
SEND_GENDER_BE,str,category,15.90,1.77,14.13,88.89,True
TRANS_VALUE,float64,float64,14.13,14.13,0.00,0.00,False
TRANS_NUM_CARD_FEWF,int64,int64,14.13,14.13,0.00,0.00,False
NID_ALPHA,int64,int32,14.13,7.07,7.07,50.00,True
SEND_LONG_REGISTER,float64,float32,14.13,7.07,7.07,50.00,True



----------------------------------------------------------------------------------------------------
FEATURES CONVERTED TO CATEGORY
----------------------------------------------------------------------------------------------------
- RECEIVE_LOC_FEWF
- RECEIVE_CATEGORY_OHEWI
- SEND_GENDER_BE
- SEND_JOB_FEWF
- TRANS_WEEK_OHEWI
- SEND_NAME_FEWF

FINAL DATASET
dataset_final.parquet created successfully!
Location: /projeto_tcc_2026/data/final_dataset/dataset_final.parquet


### <span style="color:blue">SUMMARY AND SAMPLING</span> ###

In [10]:
# ============================================================
# 26. SETTINGS
# ============================================================

primary_dataset_file = Path(
    "/projeto_tcc_2026/data/initial_dataset/dataset_primary.csv"
)

final_dataset_file = Path(
    "/projeto_tcc_2026/data/final_dataset/dataset_final.parquet"
)

report_file = Path(
    "/projeto_tcc_2026/data/final_dataset/dataset_comparison_report.txt"
)


# ============================================================
# 27. CHECK IF THE COMPARISON REPORT ALREADY EXISTS
# ============================================================

if report_file.exists():

    print(
        "\ndataset_comparison_report.txt already exists."
    )

    print(
        "A new report will not be created."
    )

    print(
        "\nDisplaying the existing report:\n"
    )

    print(
        report_file.read_text(
            encoding="utf-8"
        )
    )

else:

    # ========================================================
    # 28. CHECK REQUIRED DATASETS
    # ========================================================

    if not primary_dataset_file.exists():

        raise FileNotFoundError(
            f"Primary dataset not found:\n"
            f"{primary_dataset_file}"
        )


    if not final_dataset_file.exists():

        raise FileNotFoundError(
            f"Final dataset not found:\n"
            f"{final_dataset_file}"
        )


    # ========================================================
    # 29. LOAD THE PRIMARY DATASET
    # ========================================================

    print(
        "\nLoading the primary dataset..."
    )

    primary_df = pd.read_csv(
        primary_dataset_file
    )


    primary_shape = (
        primary_df.shape
    )

    primary_columns = (
        primary_df.columns.tolist()
    )

    primary_dtypes = (
        primary_df.dtypes
        .astype(str)
        .copy()
    )


    primary_types_table = pd.DataFrame({
        "FEATURE": primary_dtypes.index,
        "TYPE": primary_dtypes.values
    })


    primary_missing_values = (
        primary_df
        .isna()
        .sum()
        .sum()
    )


    print(
        "Primary dataset loaded successfully!"
    )


    # ========================================================
    # 30. RELEASE THE PRIMARY DATASET FROM MEMORY
    # ========================================================

    del primary_df

    gc.collect()


    # ========================================================
    # 31. LOAD THE FINAL DATASET
    # ========================================================

    print(
        "\nLoading the final dataset..."
    )

    final_df = pd.read_parquet(
        final_dataset_file,
        engine="pyarrow"
    )


    final_shape = (
        final_df.shape
    )

    final_columns = (
        final_df.columns.tolist()
    )

    final_dtypes = (
        final_df.dtypes
        .astype(str)
        .copy()
    )


    final_types_table = pd.DataFrame({
        "FEATURE": final_dtypes.index,
        "TYPE": final_dtypes.values
    })


    final_missing_values = (
        final_df
        .isna()
        .sum()
        .sum()
    )


    print(
        "Final dataset loaded successfully!"
    )


    # ========================================================
    # 32. CALCULATE FILE SIZES
    # ========================================================

    primary_size_bytes = (
        primary_dataset_file
        .stat()
        .st_size
    )

    final_size_bytes = (
        final_dataset_file
        .stat()
        .st_size
    )


    primary_size_mb = (
        primary_size_bytes
        / 1024**2
    )

    final_size_mb = (
        final_size_bytes
        / 1024**2
    )


    space_saved_bytes = (
        primary_size_bytes
        - final_size_bytes
    )

    space_saved_mb = (
        space_saved_bytes
        / 1024**2
    )


    space_reduction_percentage = (

        1
        - (
            final_size_bytes
            / primary_size_bytes
        )

    ) * 100


    # ========================================================
    # 33. IDENTIFY FEATURE DIFFERENCES
    # ========================================================

    primary_only_features = [
        column
        for column in primary_columns
        if column not in final_columns
    ]


    final_only_features = [
        column
        for column in final_columns
        if column not in primary_columns
    ]


    # ========================================================
    # 34. FINAL FEATURE DESCRIPTIONS
    # ========================================================

    feature_descriptions = {

        "NID_ALPHA":
            "Unique identifier assigned to each transaction.",

        "TRANS_NUM_CARD_FEWF":
            "Credit card number feature prepared for "
            "Frequency Encoding With Fallback (FEWF).",

        "RECEIVE_LOC_FEWF":
            "Merchant or transaction receiver location feature "
            "prepared for Frequency Encoding With Fallback (FEWF).",

        "RECIVE_LOC_FEWF":
            "Merchant or transaction receiver location feature "
            "prepared for Frequency Encoding With Fallback (FEWF).",

        "RECEIVE_CATEGORY_OHEWI":
            "Transaction category feature prepared for "
            "One-Hot Encoding With Ignore (OHEWI).",

        "RECIVE_CATEGORY_OHEWI":
            "Transaction category feature prepared for "
            "One-Hot Encoding With Ignore (OHEWI).",

        "TRANS_VALUE":
            "Monetary value of the transaction.",

        "SEND_GENDER_BE":
            "Sender gender feature prepared for Binary Encoding (BE).",

        "SEND_LAT_REGISTER":
            "Registered latitude associated with the sender.",

        "SEND_LONG_REGISTER":
            "Registered longitude associated with the sender.",

        "SEND_POP_REGISTER":
            "Population of the sender's registered city.",

        "SEND_JOB_FEWF":
            "Sender occupation feature prepared for "
            "Frequency Encoding With Fallback (FEWF).",

        "RECEIVE_LAT":
            "Latitude associated with the transaction receiver "
            "or merchant.",

        "RECIVE_LAT":
            "Latitude associated with the transaction receiver "
            "or merchant.",

        "RECEIVE_LONG":
            "Longitude associated with the transaction receiver "
            "or merchant.",

        "RECIVE_LONG":
            "Longitude associated with the transaction receiver "
            "or merchant.",

        "TRANS_DAY":
            "Day of the month on which the transaction occurred.",

        "TRANS_WEEK_OHEWI":
            "Day-of-week feature prepared for "
            "One-Hot Encoding With Ignore (OHEWI).",

        "TRANS_YEAR_BE":
            "Transaction year feature prepared for "
            "Binary Encoding (BE).",

        "TRANS_MONTH_SIN":
            "Sine component of the cyclic transformation "
            "of the transaction month.",

        "TRANS_MONTH_SEN":
            "Sine component of the cyclic transformation "
            "of the transaction month.",

        "TRANS_MONTH_COS":
            "Cosine component of the cyclic transformation "
            "of the transaction month.",

        "TRANS_HOUR_SIN":
            "Sine component of the cyclic transformation "
            "of the transaction time.",

        "TRANS_HOUR_SEN":
            "Sine component of the cyclic transformation "
            "of the transaction time.",

        "TRANS_HOUR_COS":
            "Cosine component of the cyclic transformation "
            "of the transaction time.",

        "SEND_NAME_FEWF":
            "Sender full-name feature prepared for "
            "Frequency Encoding With Fallback (FEWF).",

        "SEND_AGE":
            "Sender age in decimal years, calculated from "
            "the date of birth using the defined reference date.",

        "TARGET_OMEGA":
            "Target variable indicating whether the transaction "
            "is fraudulent."
    }


    # ========================================================
    # 35. CREATE THE FINAL FEATURE SUMMARY
    # ========================================================

    feature_summary = pd.DataFrame({

        "FEATURE":
            final_columns,

        "TYPE":
            [
                str(final_df[column].dtype)
                for column in final_columns
            ],

        "DESCRIPTION":
            [
                feature_descriptions.get(
                    column,
                    "Description not defined."
                )
                for column in final_columns
            ]
    })


    # ========================================================
    # 36. CREATE A RANDOM SAMPLE OF 10 ROWS
    # ========================================================

    random_sample = (
        final_df
        .sample(
            n=10,
            random_state=42
        )
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 37. CREATE THE TEXT REPORT
    # ========================================================

    report_lines = []


    report_lines.append(
        "=" * 100
    )

    report_lines.append(
        "PRIMARY DATASET AND FINAL DATASET COMPARISON"
    )

    report_lines.append(
        "=" * 100
    )


    # --------------------------------------------------------
    # PRIMARY DATASET DATA TYPES
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "DATA TYPES - PRIMARY DATASET"
    )

    report_lines.append(
        "=" * 100
    )

    report_lines.append(
        primary_types_table.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # FINAL DATASET DATA TYPES
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "DATA TYPES - FINAL DATASET"
    )

    report_lines.append(
        "=" * 100
    )

    report_lines.append(
        final_types_table.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # GENERAL COMPARISON
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "GENERAL COMPARISON"
    )

    report_lines.append(
        "=" * 100
    )

    report_lines.append(
        f"\nPrimary dataset rows: "
        f"{primary_shape[0]}"
    )

    report_lines.append(
        f"Final dataset rows:   "
        f"{final_shape[0]}"
    )

    report_lines.append(
        f"\nPrimary dataset features: "
        f"{primary_shape[1]}"
    )

    report_lines.append(
        f"Final dataset features:   "
        f"{final_shape[1]}"
    )

    report_lines.append(
        f"\nPrimary dataset missing values: "
        f"{primary_missing_values}"
    )

    report_lines.append(
        f"Final dataset missing values:   "
        f"{final_missing_values}"
    )


    # --------------------------------------------------------
    # DISK SPACE COMPARISON
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "DISK SPACE COMPARISON"
    )

    report_lines.append(
        "=" * 100
    )

    report_lines.append(
        f"\nPrimary dataset size: "
        f"{primary_size_mb:.2f} MB"
    )

    report_lines.append(
        f"Final dataset size:   "
        f"{final_size_mb:.2f} MB"
    )

    report_lines.append(
        f"Space saved:          "
        f"{space_saved_mb:.2f} MB"
    )

    report_lines.append(
        f"Space reduction:      "
        f"{space_reduction_percentage:.2f}%"
    )


    # --------------------------------------------------------
    # FEATURES PRESENT ONLY IN THE PRIMARY DATASET
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "FEATURES ONLY IN THE PRIMARY DATASET"
    )

    report_lines.append(
        "=" * 100
    )

    if primary_only_features:

        for feature in primary_only_features:

            report_lines.append(
                f"- {feature}"
            )

    else:

        report_lines.append(
            "No exclusive features."
        )


    # --------------------------------------------------------
    # FEATURES PRESENT ONLY IN THE FINAL DATASET
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "FEATURES ONLY IN THE FINAL DATASET"
    )

    report_lines.append(
        "=" * 100
    )

    if final_only_features:

        for feature in final_only_features:

            report_lines.append(
                f"- {feature}"
            )

    else:

        report_lines.append(
            "No exclusive features."
        )


    # --------------------------------------------------------
    # FINAL FEATURE DESCRIPTIONS
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "FINAL FEATURE SUMMARY"
    )

    report_lines.append(
        "=" * 100
    )

    report_lines.append(
        feature_summary.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # RANDOM SAMPLE
    # --------------------------------------------------------

    report_lines.append(
        "\n\n"
        + "=" * 100
    )

    report_lines.append(
        "RANDOM SAMPLE - 10 ROWS"
    )

    report_lines.append(
        "=" * 100
    )

    report_lines.append(
        random_sample.to_string(
            index=False
        )
    )


    # ========================================================
    # 38. SAVE THE TEXT REPORT
    # ========================================================

    report_text = (
        "\n".join(
            report_lines
        )
    )


    report_file.write_text(
        report_text,
        encoding="utf-8"
    )


    # ========================================================
    # 39. CONFIRM THE REPORT WAS SAVED
    # ========================================================

    if report_file.exists():

        print(
            "\nComparison report created successfully!"
        )

        print(
            "Location:",
            report_file
        )

    else:

        raise FileNotFoundError(
            "dataset_comparison_report.txt was not created."
        )


    # ========================================================
    # 40. DISPLAY DATA TYPES - PRIMARY DATASET
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "DATA TYPES - PRIMARY DATASET"
    )

    print(
        "=" * 100
    )

    display(
        primary_types_table
    )


    # ========================================================
    # 41. DISPLAY DATA TYPES - FINAL DATASET
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "DATA TYPES - FINAL DATASET"
    )

    print(
        "=" * 100
    )

    display(
        final_types_table
    )


    # ========================================================
    # 42. DISPLAY GENERAL SUMMARY
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "FINAL SUMMARY"
    )

    print(
        "=" * 100
    )


    print(
        f"Primary dataset rows: "
        f"{primary_shape[0]}"
    )

    print(
        f"Final dataset rows:   "
        f"{final_shape[0]}"
    )


    print(
        f"\nPrimary dataset features: "
        f"{primary_shape[1]}"
    )

    print(
        f"Final dataset features:   "
        f"{final_shape[1]}"
    )


    print(
        f"\nPrimary dataset size: "
        f"{primary_size_mb:.2f} MB"
    )

    print(
        f"Final dataset size:   "
        f"{final_size_mb:.2f} MB"
    )

    print(
        f"Space saved:          "
        f"{space_saved_mb:.2f} MB"
    )

    print(
        f"Space reduction:      "
        f"{space_reduction_percentage:.2f}%"
    )


    # ========================================================
    # 43. DISPLAY FEATURE DIFFERENCES
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "FEATURES ONLY IN THE PRIMARY DATASET"
    )

    print(
        "=" * 100
    )


    if primary_only_features:

        for feature in primary_only_features:

            print(
                f"- {feature}"
            )

    else:

        print(
            "No exclusive features."
        )


    print(
        "\n"
        + "=" * 100
    )

    print(
        "FEATURES ONLY IN THE FINAL DATASET"
    )

    print(
        "=" * 100
    )


    if final_only_features:

        for feature in final_only_features:

            print(
                f"- {feature}"
            )

    else:

        print(
            "No exclusive features."
        )


    # ========================================================
    # 44. DISPLAY FINAL FEATURE SUMMARY
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "FINAL FEATURE SUMMARY"
    )

    print(
        "=" * 100
    )

    display(
        feature_summary
    )


    # ========================================================
    # 45. DISPLAY A RANDOM SAMPLE OF 10 ROWS
    # ========================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        "RANDOM SAMPLE - 10 ROWS"
    )

    print(
        "=" * 100
    )

    display(
        random_sample
    )


Loading the primary dataset...
Primary dataset loaded successfully!

Loading the final dataset...
Final dataset loaded successfully!

Comparison report created successfully!
Location: /projeto_tcc_2026/data/final_dataset/dataset_comparison_report.txt

DATA TYPES - PRIMARY DATASET


,FEATURE,TYPE
0,NID,int64
1,trans_date_trans_time,str
2,cc_num,int64
3,merchant,str
4,category,str
5,amt,float64
6,first,str
7,last,str
8,gender,str
9,street,str



DATA TYPES - FINAL DATASET


,FEATURE,TYPE
0,NID_ALPHA,int32
1,TRANS_NUM_CARD_FEWF,int64
2,RECEIVE_LOC_FEWF,category
3,RECEIVE_CATEGORY_OHEWI,category
4,TRANS_VALUE,float64
5,SEND_GENDER_BE,category
6,SEND_LAT_REGISTER,float32
7,SEND_LONG_REGISTER,float32
8,SEND_POP_REGISTER,int32
9,SEND_JOB_FEWF,category



FINAL SUMMARY
Primary dataset rows: 1852394
Final dataset rows:   1852394

Primary dataset features: 23
Final dataset features:   22

Primary dataset size: 472.78 MB
Final dataset size:   49.73 MB
Space saved:          423.06 MB
Space reduction:      89.48%

FEATURES ONLY IN THE PRIMARY DATASET
- NID
- trans_date_trans_time
- cc_num
- merchant
- category
- amt
- first
- last
- gender
- street
- city
- state
- zip
- lat
- long
- city_pop
- job
- dob
- trans_num
- unix_time
- merch_lat
- merch_long
- is_fraud

FEATURES ONLY IN THE FINAL DATASET
- NID_ALPHA
- TRANS_NUM_CARD_FEWF
- RECEIVE_LOC_FEWF
- RECEIVE_CATEGORY_OHEWI
- TRANS_VALUE
- SEND_GENDER_BE
- SEND_LAT_REGISTER
- SEND_LONG_REGISTER
- SEND_POP_REGISTER
- SEND_JOB_FEWF
- RECEIVE_LAT
- RECEIVE_LONG
- TRANS_DAY
- TRANS_WEEK_OHEWI
- TRANS_YEAR_BE
- TRANS_MONTH_SIN
- TRANS_MONTH_COS
- TRANS_HOUR_SIN
- TRANS_HOUR_COS
- SEND_NAME_FEWF
- SEND_AGE
- TARGET_OMEGA

FINAL FEATURE SUMMARY


,FEATURE,TYPE,DESCRIPTION
0,NID_ALPHA,int32,Unique identifier assigned to each transaction.
1,TRANS_NUM_CARD_FEWF,int64,Credit card number feature prepared for Freque...
2,RECEIVE_LOC_FEWF,category,Merchant or transaction receiver location feat...
3,RECEIVE_CATEGORY_OHEWI,category,Transaction category feature prepared for One-...
4,TRANS_VALUE,float64,Monetary value of the transaction.
5,SEND_GENDER_BE,category,Sender gender feature prepared for Binary Enco...
6,SEND_LAT_REGISTER,float32,Registered latitude associated with the sender.
7,SEND_LONG_REGISTER,float32,Registered longitude associated with the sender.
8,SEND_POP_REGISTER,int32,Population of the sender's registered city.
9,SEND_JOB_FEWF,category,Sender occupation feature prepared for Frequen...



RANDOM SAMPLE - 10 ROWS


,NID_ALPHA,TRANS_NUM_CARD_FEWF,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI,TRANS_VALUE,SEND_GENDER_BE,SEND_LAT_REGISTER,SEND_LONG_REGISTER,SEND_POP_REGISTER,SEND_JOB_FEWF,...,TRANS_DAY,TRANS_WEEK_OHEWI,TRANS_YEAR_BE,TRANS_MONTH_SIN,TRANS_MONTH_COS,TRANS_HOUR_SIN,TRANS_HOUR_COS,SEND_NAME_FEWF,SEND_AGE,TARGET_OMEGA
0,1541145,5359543825610251,"fraud_Jenkins, Hauck and Friesen",gas_transport,59.91,M,45.780102,-111.143898,18182,"Engineer, drilling",...,18,Friday,2020,-8.660254e-01,-5.000000e-01,0.948807,-0.315856,Michael Francis,51.180000,0
1,1731582,5540636818935089,fraud_Jast-McDermott,shopping_pos,3.96,M,42.691101,-71.160500,76383,Geoscientist,...,5,Saturday,2020,-5.000000e-01,8.660254e-01,-0.998723,-0.050520,Kenneth Foster,41.419998,0
2,354660,2720894374956739,fraud_Bartoletti-Wunsch,gas_transport,51.17,F,42.597801,-82.882301,16305,"Psychologist, sport and exercise",...,15,Saturday,2019,5.000000e-01,-8.660254e-01,0.153273,-0.988184,Audrey Hickman,99.279999,0
3,1493789,6011438889172900,"fraud_Roob, Conn and Tremblay",shopping_pos,2.06,F,34.285301,-91.333603,5161,Electrical engineer,...,29,Saturday,2020,-5.000000e-01,-8.660254e-01,-0.298971,0.954262,Allison Allen,33.410000,0
4,468149,60495593109,"fraud_Kilback, Nitzsche and Leffler",travel,6.58,M,32.769901,-96.742996,1263321,Television camera operator,...,25,Thursday,2019,1.224647e-16,-1.000000e+00,-0.844756,-0.535151,Randall Dillon,83.779999,0
5,1389855,6011477612335392,"fraud_Cormier, Stracke and Thiel",entertainment,24.49,M,40.406200,-84.507599,2274,"Designer, television/film set",...,23,Thursday,2020,1.224647e-16,-1.000000e+00,-0.991340,-0.131319,Anthony Roberts,82.709999,0
6,1760907,2703186189652095,fraud_Skiles LLC,home,99.74,F,36.078800,-81.178101,3495,"Psychologist, counselling",...,11,Friday,2020,-5.000000e-01,8.660254e-01,-0.229271,0.973363,Jennifer Banks,38.490002,0
7,469448,4839615922685395,fraud_Vandervort-Funk,grocery_pos,147.12,M,39.013000,-86.545700,76,Social researcher,...,26,Friday,2019,1.224647e-16,-1.000000e+00,0.962139,0.272560,Phillip Robertson,71.330002,0
8,1835019,3553629419254918,fraud_Sporer Inc,gas_transport,46.85,F,48.340000,-122.345596,85,"Research officer, political party",...,28,Monday,2020,-5.000000e-01,8.660254e-01,0.911882,-0.410454,Sharon Johnson,42.000000,0
9,240728,4481131401752,"fraud_Wintheiser, Dietrich and Schimmel",misc_pos,33.56,M,42.284801,-71.720497,35299,English as a second language teacher,...,30,Tuesday,2019,1.000000e+00,6.123234e-17,-0.731849,-0.681466,Frank Foster,51.349998,0
